# DADA2: 01 — Corpus and index

**4 points.** Three TODOs.

By the end of this notebook, you will have indexed all the corpus in Pinecone as a couple of
thousand chunks, with every chunk knowing which filing it came from.

In [1]:
from dotenv import load_dotenv
load_dotenv()

import csv, os, json

## 1. The corpus

The corpus includes nine 10-K filings for three companies, split into three sections each. 

Let's look at what is actually on disk before we load anything.

In [2]:
for root, _, files in sorted(os.walk("corpus")):
    if files:
        print(f"{root}/  ({len(files)} files)")
        for name in sorted(files)[:3]:
            print(f"    {name}")
        if len(files) > 3:
            print(f"    ... and {len(files) - 3} more")

corpus/  (1 files)
    manifest.csv
corpus\filings/  (1 files)
    NVDA_2026_10K.htm
corpus\financials/  (3 files)
    AMD_financials.json
    MU_financials.json
    NVDA_financials.json
corpus\sections/  (27 files)
    AMD_2023_business.txt
    AMD_2023_mdna.txt
    AMD_2023_risk_factors.txt
    ... and 24 more


### The manifest

`manifest.csv` has one row per section file. It is the only place that knows a
file called `NVDA_2026_risk_factors.txt` is NVIDIA's fiscal 2026 Item 1A,
filed on a particular date, and that is the knowledge you are about to attach
to every chunk.

In [3]:
manifest = {row["file"]: row for row in csv.DictReader(open("corpus/manifest.csv"))}

print(f"{len(manifest)} files\n")
print("columns:", list(next(iter(manifest.values())).keys()), "\n")

example = manifest["NVDA_2026_risk_factors.txt"]
for key, value in example.items():
    print(f"  {key:12} {str(value)[:70]}")

27 files

columns: ['file', 'ticker', 'company', 'cik', 'section', 'fiscal_year', 'filing_date', 'form_type', 'chars', 'source_url'] 

  file         NVDA_2026_risk_factors.txt
  ticker       NVDA
  company      NVIDIA CORP
  cik          1045810
  section      risk_factors
  fiscal_year  2026
  filing_date  2026-02-25
  form_type    10-K
  chars        114960
  source_url   https://www.sec.gov/Archives/edgar/data/1045810/000104581026000021/nvd


In [4]:
# the shape of the corpus, by company and year
import collections

by_company = collections.Counter(row["ticker"] for row in manifest.values())
by_year = collections.Counter(row["fiscal_year"] for row in manifest.values())
by_section = collections.Counter(row["section"] for row in manifest.values())
print("by company:", dict(by_company))
print("by year:   ", dict(sorted(by_year.items())))
print("by section:", dict(by_section))

by company: {'NVDA': 9, 'AMD': 9, 'MU': 9}
by year:    {'2023': 3, '2024': 9, '2025': 9, '2026': 6}
by section: {'business': 9, 'risk_factors': 9, 'mdna': 9}


### Why this corpus is hard

The challenge of working with this corpus is that the three companies operate in the same market, writing about the same pressures, in
three fiscal years each.

Let's count how many files mention the same thing.

In [7]:
for phrase in ("export control", "HBM", "TSMC", "artificial intelligence"):
    hits = []
    for name in sorted(manifest):
        text = open(os.path.join("corpus/sections", name)).read().lower()
        if phrase.lower() in text:
            hits.append(name)
    print(f"{phrase:24} appears in {len(hits):2} of {len(manifest)} files")

UnicodeDecodeError: 'charmap' codec can't decode byte 0x9d in position 825: character maps to <undefined>

A search for "export controls" has a dozen files with similar passages. The embedding cannot tell them apart easily, because they really do say similar things.

But metadata can.

## 2. Loading, with the metadata attached

### TODO 1.1 — load every section with its metadata (2 pts)

Put exactly these four fields on every Document, and no more:

```
source       the filename, e.g. "NVDA_2026_risk_factors.txt"
ticker       "NVDA" | "AMD" | "MU"
fiscal_year  an int, not a string
section      "business" | "risk_factors" | "mdna"
```

Two of those get filtered on in notebook 02 and one is what the metrics score
against, so the names matter. Resist attaching the whole manifest row: every
field you add is stored on every chunk, and only the fields something queries
earn their place.

Please note `fiscal_year` has to be an `int`.
Pinecone stores `"2026"` and `2026` as different things, a filter for the
number will not match the string, and nothing will warn you. You will just
get zero results in notebook 02 and no explanation.

Study the documentation for LangChain's [TextLoader()](https://docs.langchain.com/oss/javascript/integrations/document_loaders/file_loaders/text)

**Steps**, one per line of the loop:

1. Loop over the filenames in `manifest`. It is keyed by filename.
2. Load the file. `TextLoader(path).load()` returns a list with one `Document` in it for a text file. The path is `corpus/sections/` plus the name.
3. Replace the Document's metadata. The loader only put a path there. Build a new dict with the four fields, reading `ticker`, `fiscal_year` and `section` from `manifest[name]`. `source` is the bare filename. Cast the year as int.
4. Append it to `documents`.

Then run the check cell below. All four asserts should pass before you go on.


In [ ]:
from langchain_community.document_loaders import TextLoader

documents = []
# TODO 1.1
# your code here


print(f"{len(documents)} documents")
if documents:
    print("metadata on the first:", documents[0].metadata)
    print("\ncontent on the first:", documents[0].page_content[:100])

In [ ]:
# Check it before you go on. All four of these should pass.
assert len(documents) == 27, f"expected 27 documents, got {len(documents)}"

keys = {tuple(sorted(d.metadata)) for d in documents}
assert keys == {("fiscal_year", "section", "source", "ticker")}, \
    f"metadata keys should be exactly those four, found {keys}"

assert all(isinstance(d.metadata["fiscal_year"], int) for d in documents), \
    "fiscal_year must be an int, not a string"

assert {d.metadata["ticker"] for d in documents} == {"NVDA", "AMD", "MU"}

print("metadata looks right")

### TODO 1.2 — Chunking with metadata (1 pt)

**Steps:**

1. Make a `RecursiveCharacterTextSplitter` with the chunk_size and chunk_overlap constants from config.py
2. Use the splitter's `split_documents(documents)`. Not `split_text`: that takes strings, returns strings, and the metadata will be lost.`split_documents` takes Documents and copies each one's metadata onto its pieces.

Then the check cell: `chunks[0].metadata` should still have all four fields.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rag import config

# TODO 1.2
chunks = []
# your code here

print(f"{len(documents)} documents became {len(chunks)} chunks")

In [ ]:
# Did the metadata survive the split?
assert chunks, "no chunks"
assert chunks[0].metadata.get("source"), "chunks lost their metadata"
assert all(c.metadata.get("ticker") for c in chunks), "some chunks have no ticker"
print("sample chunk metadata:", chunks[0].metadata)
print()
print(chunks[0].page_content[:300])

### A thing worth looking at

Print a few consecutive chunks from one file and look at where the splits
landed. Sentences are cut in half. Some risk factors have their heading in one
chunk and their substance in the next. That is the cross-reference break from
W05, and nothing in notebook 02 repairs it. Look once, so you know the
ceiling.

In [ ]:
same_file = [c for c in chunks if c.metadata["source"] == "NVDA_2026_risk_factors.txt"]
for c in same_file[3:6]:
    print(f"--- {c.metadata['source']} ({len(c.page_content)} chars) ---")
    print(c.page_content[:260])
    print("   ...")
    print(c.page_content[-160:])
    print()

## 3. Embedding

`config.embeddings()` returns the embedding model. `EMBEDDINGS` in
`rag/config.py` defaults to `cloudflare`: `qwen3-embedding-0.6b`, hosted on
Workers AI, through the same token the decision model uses in notebook 03.

It is the right default for three reasons. It is free: the whole corpus costs
about 400 of the 10,000 neurons a day. It is quick: about four minutes. And
the app you deploy in notebook 03 can use it as it is, because on Render the
only thing available is an API key.

The backup is the same model run locally. Put `EMBEDDINGS='ollama'` in `.env`,
restart the kernel, and `ollama pull qwen3-embedding:0.6b`. Its vectors match
Cloudflare's to four decimal places, so both read and write the same index,
`dada2-1024`, and you can switch between them without re-indexing.

The lecture notebooks used Gemini's embedding model, and `EMBEDDINGS='gemini'`
still works. Know what it costs before you pick it: the free tier allows
1,000 embeddings a day, and this corpus is 2,124 chunks. Indexing it with
Gemini needs the paid tier.

In [ ]:
from rag import config

embeddings = config.embeddings()
probe = embeddings.embed_query("export control restrictions")
print(f"{config.EMBEDDINGS}: {config.EMBED_MODEL}, {len(probe)} dimensions")

### TODO 1.3 — index to Pinecone (1 pt)

Save every chunk with its metadata in Pinecone.

This takes a few minutes. Print progress so you can tell slow from hung Upload in batches with
`vector_store.add_documents`.

**Steps:**

1. Store the chunks in your Pinecone index using the technique from the W5a lecture (first add a new index and use its name here)

It takes a few minutes. The checkpoint cell below tells you whether the metadata made it.


In [ ]:
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

# 3072 is the number of embeddings dimensions
INDEX_NAME = config.index_name(len(probe))
# TODO 1.3
vector_store = None
# your code here

print("indexed")

### Checkpoint — does the metadata round-trip?

This is the cell that decides whether notebook 02 is going to work. An
unfiltered search should return passages from several companies. The same
search filtered to one filing should return passages from that filing only.

If the filtered search comes back empty, the usual cause is `fiscal_year`
stored as a string.

In [ ]:
query = "export control restrictions on data center GPUs"

print("unfiltered:")
for hit in vector_store.similarity_search(query, k=5):
    print("   ", hit.metadata["source"])

print()
print("filtered to NVDA fiscal 2026:")
where = {"ticker": {"$eq": "NVDA"}, "fiscal_year": {"$eq": 2026}}
filtered = vector_store.similarity_search(query, k=5, filter=where)
for hit in filtered:
    print("   ", hit.metadata["source"])

assert filtered, "filtered search returned nothing. Is fiscal_year an int?"
assert all(h.metadata["source"] == "NVDA_2026_risk_factors.txt"
           or h.metadata["ticker"] == "NVDA" for h in filtered)
print()
print("the filter works")

The unfiltered list mixes companies and years. Some of those passages are the right topic in the wrong filing.

### Optional, 0 points — the filing that does not parse

`corpus/filings/` holds one raw 10-K as it comes off EDGAR. Everything else
in this corpus was extracted for you by `scripts/fetch_corpus.py`. If you want
to see why that script exists, try pulling Item 1A out of this file yourself.

`BSHTMLLoader` needs `bs_kwargs={"features": "html.parser"}` here unless you
have lxml installed. Expect inline XBRL tags in the middle of sentences,
section headings that appear in the table of contents before they appear as
headings, and `Item 1A.` written four different ways. 

This is what document loading actually looks like outside a course assignment.

In [ ]:
# optional, nothing is graded here
from langchain_community.document_loaders import BSHTMLLoader

raw = BSHTMLLoader("corpus/filings/NVDA_2026_10K.htm",
                   bs_kwargs={"features": "html.parser"}).load()[0]
text = " ".join(raw.page_content.split())
print(f"{len(text):,} characters")
print(text)